# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Lane: **Refresh / Content Opportunity Scoring** — a binary decision (does this page need a refresh?) with a stated reason per page. Three methods from the menu fit this directly, so all three are tried rather than picking one by assumption:

- **Decision Tree** — can combine multiple signals at once (unlike the single-feature baseline), stays interpretable at a shallow depth, and naturally handles the mixed numeric/categorical feature set without scaling.
- **Random Forest** — the natural next step from a single tree: averages many trees to reduce the variance a single deep tree would have, while still supporting feature-importance inspection.
- **Logistic Regression** — the linear-boundary control case. If this underperforms the tree-based methods, that's itself informative: it suggests the relationship between features and the label isn't well described by a straight line.

**What's deliberately not used:** clustering and permutation importance weren't the right fit here — this is a direct classification/ranking task with a clear label already, not an unsupervised grouping problem, and feature importance from the tree/forest models already gives interpretability without a separate permutation step. Gradient Boosting was left out to keep the model set aligned with what's directly comparable to the already-established baseline methodology, not because it's unsafe here.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

# Same population as the Week-4 baseline: fact_content_daily_performance, March 2026,
# split into first/second half of the month, only pages with real search data in both.
page_level_full = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_impressions) as avg_impressions_h1,
               AVG(gsc_clicks) as avg_clicks_h1,
               AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) as ctr_h1,
               AVG(gsc_avg_position) as avg_position_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) as active_days_h1
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date <= '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT content_hash_id, AVG(gsc_avg_position) as avg_position_h2
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date > '2026-03-15'
        GROUP BY content_hash_id
    )
    SELECT f.*, s.avg_position_h2
    FROM first_half f
    JOIN second_half s ON f.content_hash_id = s.content_hash_id
""").df()

page_level_full = page_level_full[page_level_full["avg_position_h1"] > 0].copy()  # exclude the position=0 artifact
page_level_full["needs_refresh"] = (page_level_full["avg_position_h2"] > page_level_full["avg_position_h1"]).astype(int)

print(f"Pages: {len(page_level_full):,}  |  Clients: {page_level_full['client_hash_id'].nunique()}")
print(f"Base rate (needs_refresh=1): {page_level_full['needs_refresh'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 140,599  |  Clients: 43
Base rate (needs_refresh=1): 0.538


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


**Grouped by `client_hash_id`, 5-fold GroupKFold** — the same split logic as the Week-4 baseline, on purpose: a fair model-vs-baseline comparison requires identical folds, not just the same dataset. Pages from the same client are likely to share client-level patterns (site structure, content strategy, existing search presence), so a plain random split would let that shared signal leak between train and test through client identity rather than genuine feature signal. Grouping ensures every client's pages are used for testing exactly once, never split across train and test in the same fold.

**Not time-aware, because it doesn't need to be here** — the leakage risk this study cares about is *client* leakage, not *temporal* leakage: the label itself is already built from a strict first-half/second-half time split (Section 1), so the time-ordering discipline is enforced at the feature/label construction stage, before the model ever sees the data — not at the cross-validation stage.

In [2]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
fold_sizes = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(page_level_full, groups=page_level_full["client_hash_id"])):
    te = page_level_full.iloc[te_idx]
    fold_sizes.append((fold, len(te), te["client_hash_id"].nunique(), te["needs_refresh"].mean()))
    print(f"Fold {fold}: {len(te):,} test rows, {te['client_hash_id'].nunique()} clients, "
          f"label balance {te['needs_refresh'].mean():.3f}")

Fold 0: 28,111 test rows, 8 clients, label balance 0.608
Fold 1: 28,112 test rows, 8 clients, label balance 0.652
Fold 2: 28,110 test rows, 10 clients, label balance 0.450
Fold 3: 28,149 test rows, 6 clients, label balance 0.449
Fold 4: 28,117 test rows, 11 clients, label balance 0.534


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Baseline: the Week-4 rank-normalized `avg_position_h1` heuristic, re-derived fold-by-fold (fit on each fold's train split only, never the full dataset, to keep the comparison honest). Metric: AUC, the same threshold-free ranking metric the baseline was evaluated on — plus a paired significance test against the baseline per fold, since "slightly higher AUC" and "significantly higher AUC" are different claims.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy import stats

feature_cols = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]

results = {"baseline": [], "tree": [], "rf": [], "logreg": []}

for fold, (tr_idx, te_idx) in enumerate(gkf.split(page_level_full, groups=page_level_full["client_hash_id"])):
    tr = page_level_full.iloc[tr_idx]
    te = page_level_full.iloc[te_idx]

    # Baseline, re-fit on this fold's train split only
    sorted_pos = np.sort(tr["avg_position_h1"].values)
    ranks = np.searchsorted(sorted_pos, te["avg_position_h1"].values, side="right") / len(sorted_pos)
    base_score = 1 - ranks
    results["baseline"].append(roc_auc_score(te["needs_refresh"], base_score))

    Xtr, ytr = tr[feature_cols], tr["needs_refresh"]
    Xte, yte = te[feature_cols], te["needs_refresh"]

    tree = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced").fit(Xtr, ytr)
    results["tree"].append(roc_auc_score(yte, tree.predict_proba(Xte)[:, 1]))

    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1).fit(Xtr, ytr)
    results["rf"].append(roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]))

    scaler = StandardScaler()
    lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42).fit(scaler.fit_transform(Xtr), ytr)
    results["logreg"].append(roc_auc_score(yte, lr.predict_proba(scaler.transform(Xte))[:, 1]))

print("=== Model vs baseline: mean CV AUC ===")
for name, scores in results.items():
    print(f"{name:10s} mean={np.mean(scores):.4f}  std={np.std(scores):.4f}")

print("\n=== Paired significance vs baseline (fold-by-fold) ===")
for name in ["tree", "rf", "logreg"]:
    t, p = stats.ttest_rel(results[name], results["baseline"])
    diff = np.mean(results[name]) - np.mean(results["baseline"])
    verdict = "beats baseline (p<0.05)" if p < 0.05 and diff > 0 else "does not beat baseline"
    print(f"{name:10s} diff={diff:+.4f}  p={p:.3f}  -> {verdict}")

=== Model vs baseline: mean CV AUC ===
baseline   mean=0.6348  std=0.0485
tree       mean=0.6272  std=0.0381
rf         mean=0.6349  std=0.0430
logreg     mean=0.6179  std=0.0378

=== Paired significance vs baseline (fold-by-fold) ===
tree       diff=-0.0076  p=0.240  -> does not beat baseline
rf         diff=+0.0001  p=0.986  -> does not beat baseline
logreg     diff=-0.0169  p=0.093  -> does not beat baseline


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Rather than stop at the AUC table above, the actual mistakes were inspected directly: among pages that were true decliners but the model ranked lowest (i.e. missed), how do they differ from pages correctly identified as stable?

In [5]:
# Use the last fold's random forest predictions for a concrete error look (any fold works;
# this one is simply the one already in memory from the loop above)
te_last = page_level_full.iloc[te_idx].copy()
te_last["rf_score"] = rf.predict_proba(te_last[feature_cols])[:, 1]

true_decliners = te_last[te_last["needs_refresh"] == 1]
missed = true_decliners.nsmallest(int(len(true_decliners) * 0.2), "rf_score")
correctly_stable = te_last[te_last["needs_refresh"] == 0]

print(f"Missed true decliners - avg active_days_h1: {missed['active_days_h1'].mean():.1f}")
print(f"Correctly-identified stable pages - avg active_days_h1: {correctly_stable['active_days_h1'].mean():.1f}")
print("\nInterpretation: unlike the baseline's error pattern reported in the paper (where the")
print("baseline's misses skewed toward pages with LONGER history), this Random Forest's misses")
print("on this fold actually skew slightly the other way - missed decliners have somewhat FEWER")
print("active days (9.8) than correctly-identified stable pages (11.7). This is a modest gap, not")
print("a strong pattern, and it's evaluated on a single fold rather than pooled across all five -")
print("read as a directional observation about this model's error shape, not a confirmed finding")
print("on the scale of the baseline's pooled, whole-population blind spot in the paper.")

# Feature importance - what the forest actually leans on
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nRandom Forest feature importances:")
print(importances)

Missed true decliners - avg active_days_h1: 9.8
Correctly-identified stable pages - avg active_days_h1: 11.7

Interpretation: unlike the baseline's error pattern reported in the paper (where the
baseline's misses skewed toward pages with LONGER history), this Random Forest's misses
on this fold actually skew slightly the other way - missed decliners have somewhat FEWER
active days (9.8) than correctly-identified stable pages (11.7). This is a modest gap, not
a strong pattern, and it's evaluated on a single fold rather than pooled across all five -
read as a directional observation about this model's error shape, not a confirmed finding
on the scale of the baseline's pooled, whole-population blind spot in the paper.

Random Forest feature importances:
avg_position_h1       0.782690
avg_impressions_h1    0.075649
active_days_h1        0.057207
avg_clicks_h1         0.047550
ctr_h1                0.036904
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.